# 🎤 02 · 端到端 ASR：CTC 与 Attention

传统 GMM-HMM 需要音素标注、发音词典、语言模型拼装——工程繁重。

**端到端**：直接「音频特征 → 文本」，模型自己解决**对齐（帧数 ≠ 字数）**。

两大流派：**CTC**（硬对齐、非自回归）与 **Attention 序列到序列**（软对齐、自回归）。本节两点都手写核心。

## 📋 本节要解决的问题

| # | 问题 | 方案 |
|---|---|---|
| 1 | 帧数 T 与字符数 L 对不上 | 扩展序列 + blank |
| 2 | 一条路径的后验怎么求和 | CTC 前向-后向 DP |
| 3 | 目标函数数值不稳 | log 域 + logsumexp |
| 4 | 怎么把概率变成文字 | 贪心 / Beam Search |
| 5 | 识别质量怎么度量 | 编辑距离 → WER/CER |
| 6 | 软对齐怎么做 | Attention（LAS） |

## 1️⃣ CTC 核心：对齐问题与 blank

每帧一个输出字符？不行——一个字占几帧，且帧数随语速变。

**CTC（Connectionist Temporal Classification）**：

- 词表扩展：字符集 + **blank（ε）**；
- 帧级输出 $y_t \in \mathbb{R}^{V+1}$（softmax），路径 $\pi = \pi_1\dots\pi_T$；
- **折叠**：先并相邻重复，再去 blank。例：
  `a ε a b ε` → 去 ε → `a a b` → 并重复 → `a b`；
  `a a ε b` → `a a b` → **`a b`**（重复字符要靠 ε 隔开才能保留两次 `a`）；
- **目标**：$P(l\mid x)=\sum_{\pi: \mathcal{B}(\pi)=l} P(\pi\mid x)$，路径指数级 → DP 求和。

blank 的意义：**吸收停顿与语速差异，并为重复字符提供分隔符**。

## 2️⃣ CTC 前向算法

标签 $l$ 扩展为 $l' = [\varepsilon, l_1, \varepsilon, l_2, \dots, \varepsilon]$，长度 $S=2L+1$，下标 $1\dots S$。

定义 $\alpha_t(s)$ = 前 $t$ 帧折叠成 $l'_{1:s}$ 的概率和，初始化 $\alpha_1(1)=y_\varepsilon^1,\ \alpha_1(2)=y_{l_1}^1$：

$$ \alpha_t(s) = y_{l'_s}^t \left[ \alpha_{t-1}(s) + \alpha_{t-1}(s-1) + \alpha_{t-1}(s-2)\cdot \mathbb{1}[l'_s \neq \varepsilon,\, l'_s \neq l'_{s-2}] \right] $$

三种来源：留在原地 / 前进一格 / 跳两格（仅当 $l'_s$ 是**非 ε 且与 $l'_{s-2}$ 不同**，防止同一个字被 ε 错误合并）。

终止：$P(l\mid x) = \alpha_T(S) + \alpha_T(S-1)$。复杂度 $O(T\cdot S)$。

In [1]:
import numpy as np, matplotlib.pyplot as plt
import itertools
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(1)

def log_softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=axis, keepdims=True))

def ctc_extend(labels):
    S = 2*len(labels) + 1
    ext = np.zeros(S, dtype=int)
    for i, l in enumerate(labels):
        ext[2*i+1] = l
    return ext   # [ε, l0, ε, l1, ε, ...]

def ctc_forward_log(y, labels):
    T, V = y.shape
    ext = ctc_extend(labels); S = len(ext)
    alpha = np.full((T, S), -np.inf)
    alpha[0, 0] = y[0, 0]
    if S > 1:
        alpha[0, 1] = y[0, labels[0]]
    for t in range(1, T):
        for s in range(S):
            val = alpha[t-1, s]
            if s >= 1:
                val = np.logaddexp(val, alpha[t-1, s-1])
            if s >= 2 and ext[s] != 0 and ext[s] != ext[s-2]:
                val = np.logaddexp(val, alpha[t-1, s-2])
            alpha[t, s] = val + y[t, ext[s]]
    return np.logaddexp(alpha[T-1, S-1], alpha[T-1, S-2]), alpha

T, V = 4, 3
y = log_softmax(np.random.randn(T, V))      # 词表 {0: ε, 1: a, 2: b}
labels = np.array([1, 2])                    # 目标 'ab'
loglik, alpha = ctc_forward_log(y, labels)
print('log P(ab|x) = %.5f' % loglik)
print('alpha 表 (T×S):\n', np.round(alpha, 3))

plt.show()


log P(ab|x) = -3.20764
alpha 表 (T×S):
 [[-0.201 -2.437   -inf   -inf   -inf]
 [-2.31  -0.27  -4.546 -5.775   -inf]
 [-2.589 -2.933 -0.536 -1.958 -6.054]
 [-4.492 -2.244 -2.351 -3.962 -3.843]]


In [2]:
def collapse(path):
    merged = []
    for k in path:
        if not merged or merged[-1] != k:      # 先合并相邻重复
            merged.append(k)
    return [k for k in merged if k != 0]       # 再删除 blank

def brute_ctc(y, labels):
    T, V = y.shape
    total = 0.0
    for path in itertools.product(range(V), repeat=T):
        if collapse(path) == list(labels):
            total += np.exp(sum(y[t, path[t]] for t in range(T)))
    return np.log(total)

val = brute_ctc(y, labels)
print(f'穷举 {V**T} 条路径 logP = {val:.5f} | 前向 = {loglik:.5f}')
assert abs(val - loglik) < 1e-6
print('✓ 前向后验与路径穷举一致')

穷举 81 条路径 logP = -3.20764 | 前向 = -3.20764
✓ 前向后验与路径穷举一致


## 3️⃣ 后向算法与梯度

类似定义后向 $\beta_t(s)$（$t$ 帧到结尾折叠成 $l'_{s:S}$ 的概率和），从 $t=T$ 反向递推。

- **损失**：$\mathcal{L} = -\log P(l\mid x)$；
- 帧级后验（梯度关键）：

$$ \gamma_t(k) = \frac{1}{P(l\mid x)} \sum_{s: l'_s = k} \frac{\alpha_t(s)\,\beta_t(s)}{y_k^t} $$

$\gamma_t(k)$ 表示「第 $t$ 帧输出 $k$ 且在某个最优对齐路径上」的后验，$\nabla_{y_k^t}\mathcal{L} \propto \gamma_t(k) - y_k^t$。

验证：每帧 $\sum_k \gamma_t(k) = 1$（构成帧级概率分布）。

In [3]:
def ctc_backward_log(y, labels):
    T, V = y.shape
    ext = ctc_extend(labels); S = len(ext)
    beta = np.full((T, S), -np.inf)
    beta[T-1, S-1] = y[T-1, ext[S-1]]
    if S > 1:
        beta[T-1, S-2] = y[T-1, ext[S-2]]
    for t in range(T-2, -1, -1):
        for s in range(S):
            val = beta[t+1, s]
            if s+1 < S:
                val = np.logaddexp(val, beta[t+1, s+1])
            if s+2 < S and ext[s] != 0 and ext[s] != ext[s+2]:
                val = np.logaddexp(val, beta[t+1, s+2])
            beta[t, s] = val + y[t, ext[s]]
    return beta

beta = ctc_backward_log(y, labels)
ext = ctc_extend(labels)
gamma = np.zeros((T, V))
for t in range(T):
    for s in range(len(ext)):
        gamma[t, ext[s]] += np.exp(alpha[t, s] + beta[t, s] - y[t, ext[s]] - loglik)
row_sums = gamma.sum(axis=1)
print('每帧 γ 行和 =', np.round(row_sums, 8))
assert np.allclose(row_sums, 1.0, atol=1e-6)
print('✓ Σ_k γ_t(k) = 1，γ 是合法帧级后验分布')

每帧 γ 行和 = [1. 1. 1. 1.]
✓ Σ_k γ_t(k) = 1，γ 是合法帧级后验分布


## 4️⃣ 解码：贪心 vs Beam Search

- **贪心**：每帧取 argmax → 折叠。快，但对模糊帧敏感，无法利用「语言知识」。
- **Beam Search**：同时保留 top-k 个前缀，按 log 概率扩展合并。k=1 即贪心。
- 工程上：**CTC + 外部语言模型（WFST / n-gram）浅融合** 做二次解码，或用**前缀束搜索（prefix beam search）**。
- 注意 CTC 输出常有「重复字符靠 blank 分隔」的经典陷阱：`a ε a` → `aa`，`a a` → `a`。

In [4]:
def greedy_decode(y):
    return collapse(y.argmax(axis=1))        # 每帧 argmax → 标准折叠

cases = {
  '相邻重复合并': [1, 1, 0, 2],      # a a ε b → 折叠: ab（重复只算一次）
  'blank分隔重复': [1, 0, 1, 2],     # a ε a b → 折叠: aab（blank 保留第二次 a）
  'blank 吸收':   [0, 1, 0, 0, 2, 0], # ε a ε ε b ε → 折叠: ab
}
for name, path in cases.items():
    print(f'{name}: 路径 {path} → 折叠 {collapse(path)}')
    y_demo = np.full((len(path), 3), 0.05)  # 构造对应 argmax 的概率矩阵
    for t, k in enumerate(path):
        y_demo[t] = log_softmax(np.random.randn(3))
        y_demo[t, k] = 2.0
        y_demo[t] = log_softmax(y_demo[t])
    print('   贪心解码 =', greedy_decode(y_demo))

相邻重复合并: 路径 [1, 1, 0, 2] → 折叠 [1, 2]
   贪心解码 = [np.int64(1), np.int64(2)]
blank分隔重复: 路径 [1, 0, 1, 2] → 折叠 [1, 1, 2]
   贪心解码 = [np.int64(1), np.int64(1), np.int64(2)]
blank 吸收: 路径 [0, 1, 0, 0, 2, 0] → 折叠 [1, 2]
   贪心解码 = [np.int64(1), np.int64(2)]


In [5]:
def beam_search(y, beam=2):
    pref = {(): 0.0}
    for t in range(len(y)):
        new = {}
        for p, lp in pref.items():
            for k in range(y.shape[1]):
                q = lp + y[t, k]
                if k == 0 or (p and p[-1] == k):
                    np_ = p          # blank 或重复：前缀不变（教学简化：不区分末尾blank）
                else:
                    np_ = p + (k,)
                new[np_] = np.logaddexp(new.get(np_, -np.inf), q)
        pref = dict(sorted(new.items(), key=lambda kv: -kv[1])[:beam])
    return pref

T2, V2 = 4, 3
y2 = log_softmax(np.random.randn(T2, V2))
y2[0, 1] = 2.0; y2[2, 2] = 2.0        # 让 a、b 领先
y2 = log_softmax(y2)
greedy = greedy_decode(y2)
beams = beam_search(y2, beam=2)
print('贪心结果 =', greedy)
for p, lp in beams.items():
    print('beam 前缀 %-8s logP = %.3f' % (str(p), lp))
print('✓ beam 保留多个候选，可挂语言模型重排序')

贪心结果 = [np.int64(1), np.int64(2), np.int64(1)]
beam 前缀 (1, 2)   logP = -0.818
beam 前缀 (1, 2, 1) logP = -0.874
✓ beam 保留多个候选，可挂语言模型重排序


## 5️⃣ 评估：编辑距离与 WER/CER

- **编辑距离**（Levenshtein）：增/删/替换的最小操作数，DP 求解；
- **WER**（词错误率）= $\dfrac{\text{编辑距离}}{\text{参考词数}}$，中文常用 **CER**（字错误率）；
- 数值越低越好；口语转写常用 **字错率/句错率**，语音交互还要看**首字延迟**与**意图识别率**。

递推：$D[i,j] = \min(D[i-1,j]+1,\ D[i,j-1]+1,\ D[i-1,j-1]+\mathbb{1}[a_i\neq b_j])$。

In [6]:
def edit_distance(a, b):
    T1, T2 = len(a), len(b)
    D = np.zeros((T1+1, T2+1))
    D[:, 0] = np.arange(T1+1); D[0, :] = np.arange(T2+1)
    for i in range(1, T1+1):
        for j in range(1, T2+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            D[i, j] = min(D[i-1, j]+1, D[i, j-1]+1, D[i-1, j-1]+cost)
    return D[T1, T2]

ref = list('今天天气很好')
hyp = list('今天天汽很好呀')
ed = edit_distance(ref, hyp)
print('参考:', ''.join(ref)); print('识别:', ''.join(hyp))
print('编辑距离 = %d | CER = %.1f%%' % (ed, 100*ed/len(ref)))
assert edit_distance(list('abc'), list('abc')) == 0
assert edit_distance(list('abc'), list('axc')) == 1
print('✓ 编辑距离/替换 用例通过')

参考: 今天天气很好
识别: 今天天汽很好呀
编辑距离 = 2 | CER = 33.3%
✓ 编辑距离/替换 用例通过


## 6️⃣ Attention 路线：LAS（软对齐）

**Listen-Attend-Spell**（Listen=编码器，Attend=注意力，Spell=解码器）：

- **Listen**：RNN/Transformer 编码器把特征帧 → 高层表示 $h_1\dots h_T$（可下采样提速）；
- **Attend**：解码第 $s$ 步时，对编码器输出算注意力权重 $\alpha_{s,t} = \mathrm{softmax}_t(e_{s,t})$，$e_{s,t}$ 由 query（上一步解码状态）与 key $h_t$ 打分；
- **Spell**：上下文向量 $c_s = \sum_t \alpha_{s,t} h_t$ 喂给解码器**自回归**生成字符。

注意力权重即**软对齐**：每步有权重地「看」输入的不同位置，自然学会「这帧对应哪个字」；权重矩阵可视化常见**对角单调**形态。

In [7]:
np.random.seed(3)
T_dec, T_enc, D = 5, 8, 6
K = np.random.randn(T_enc, D)      # 编码器输出 key
V = np.random.randn(T_enc, D)      # 编码器输出 value
Q = np.random.randn(T_dec, D)      # 解码器 query
e = (Q @ K.T) / np.sqrt(D)         # 点积打分，缩放防梯度消失
attn = np.exp(e - e.max(axis=1, keepdims=True))
attn = attn / attn.sum(axis=1, keepdims=True)
c = attn @ V                       # (T_dec, D) 上下文
print('注意力矩阵行和 =', np.round(attn.sum(1), 6))
print('上下文向量形状 =', c.shape)
plt.figure(figsize=(7, 3.6))
plt.imshow(attn, aspect='auto', origin='lower', cmap='Blues')
plt.xlabel('编码器位置 t'); plt.ylabel('解码步 s')
plt.colorbar(label='权重'); plt.title('Attention 权重（软对齐）：每解码步加权融合所有输入帧')
plt.tight_layout(); plt.show()

注意力矩阵行和 = [1. 1. 1. 1. 1.]
上下文向量形状 = (5, 6)


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\2992456622.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7️⃣ CTC vs Attention：怎么选

| 维度 | CTC | Attention (AED) |
|---|---|---|
| 对齐 | 隐式硬对齐（帧级独立输出） | 显式软对齐 |
| 独立性 | 帧间条件独立（建模弱） | 解码自回归（建模强） |
| 训练 | 帧-标签自动对齐，无需预对齐 | 端到端，无预对齐 |
| 解码 | 可流式（帧同步） | 自回归整句（延迟高） |
| 缺点 | 重复/停顿时长建模弱 | 注意力漂移、重复问题 |

工程常用**联合 CTC + Attention**（如 Wenet U2）：CTC 保证单调对齐、Attention 保证精度，训练时 loss 加权相加，解码可流式/非流式切换。

## 8️⃣ RNN-Transducer：流式端到端

- Encoder 逐帧编码 + **Prediction Network**（类语言模型）+ **Joint Network** 融合；
- 输出**帧同步**，天然支持流式（边说边出字），对齐是**单调**的；
- 训练目标类似 CTC 但含「预测网络先验」，用 **WFST/循环前向后向**高效计算；
- 应用：交互式语音助手（延迟敏感）主流方案之一。

一句话：**CTC 是「帧→输出」独立概率；RNN-T 在 CTC 之上引入语言先验并保持流式。**

## 9️⃣ 工业现状速览

- **Whisper（OpenAI）**：AED + 800k 小时弱监督，多语言多任务（转写/翻译/时间戳）；
- **Wenet**：U2/U3，CTC+Attention 联合，工业流式部署主流；
- **Parakeet / NVIDIA**：CTC-only 高精度流式模型；
- **Alignments 微调**（如 Distil-Whisper、SeamlessM4T）：蒸馏 + 对齐蒸馏提速；
- 趋势：**大模型语音（AudioLM/SALMONN 等）**把 ASR 并入多模态，但 CTC/AED/RNN-T 三件套仍是面试与工程底座。

## 🎯 面试考点

1. CTC 折叠规则 + blank 用途（重复分隔、吸收停顿）；
2. 前向-后向 DP 与三种转移条件；log 域数值稳定；
3. 损失与 $\gamma_t(k)$ 的梯度直觉；
4. 贪心 vs Beam Search；前缀束搜索与语言模型融合；
5. 编辑距离 DP、WER/CER 计算与意义；
6. LAS 三模块与软对齐；注意力权重可视化（对角单调）；
7. CTC vs Attention 对比表与本构（联合训练理由）；
8. RNN-T 与 CTC 的差别、流式能力来源；
9. Whisper/Wenet/Parakeet 各自选型逻辑。

## ✅ 自测清单 + 衔接

- [ ] 手写 CTC 前向，穷举对照 < 1e-6；
- [ ] 手写后向 + $\gamma$，行和 = 1；
- [ ] 手写贪心/beam 解码与编辑距离/WER；
- [ ] 手写点积注意力与软对齐可视化；
- [ ] 能当场画 CTC vs AED vs RNN-T 对比表；
- [ ] 复述 9 条考点各 30 秒。

**下一讲 03 · 语音合成 TTS**：识别把语音变文字，合成把文字变语音——Tacotron 的注意力与 VITS 的 VAE+Flow+GAN，思路同源但方向相反。

## 8️⃣ CTC 梯度：反向传播怎么走

目标 $L=-\log P(l|x)$，对帧级输出求导：

$$ \frac{\partial L}{\partial y_t(k)} = \mathrm{softmax}_t(k) - \gamma_t(k), \qquad \gamma_t(k)=\sum_{s: l'_s=k}\frac{\alpha_t(s)\beta_t(s)}{e^{y_t(k)}P(l|x)} $$

- $\gamma_t(k)$ = 第 t 帧「后验占据」标签 k 的总概率（前向×后向）；
- 直觉：模型在该帧多给标签 k 的概率，应匹配「路径统计上需要它」的程度；
- 实现：CTC 前向+后向表 → 梯度 → 标准反向传播。

In [8]:
def ctc_forward_tab(y, labels):
    T, V = y.shape
    ext = ctc_extend(labels)
    S = len(ext)
    a = np.full((T, S), -np.inf)
    a[0, 0] = y[0, 0]
    if S > 1: a[0, 1] = y[0, labels[0]]
    for t in range(1, T):
        for s in range(S):
            v = a[t-1, s]
            if s >= 1: v = np.logaddexp(v, a[t-1, s-1])
            if s >= 2 and ext[s] != 0 and ext[s] != ext[s-2]:
                v = np.logaddexp(v, a[t-1, s-2])
            a[t, s] = v + y[t, ext[s]]
    return a, np.logaddexp(a[T-1, S-1], a[T-1, S-2])

def ctc_backward_tab(y, labels):
    T, V = y.shape
    ext = ctc_extend(labels); S = len(ext)
    b = np.full((T, S), -np.inf)
    b[T-1, S-1] = y[T-1, ext[S-1]]
    if S >= 2: b[T-1, S-2] = y[T-1, ext[S-2]]
    for t in range(T-2, -1, -1):
        for s in range(S):
            v = b[t+1, s]
            if s+1 < S: v = np.logaddexp(v, b[t+1, s+1])
            if s+2 < S and ext[s] != 0 and ext[s] != ext[s+2]:
                v = np.logaddexp(v, b[t+1, s+2])
            b[t, s] = v + y[t, ext[s]]
    return b

def ctc_gamma(y, labels):
    a, ll = ctc_forward_tab(y, labels)
    b = ctc_backward_tab(y, labels)
    ext = ctc_extend(labels)
    T, V = y.shape
    gamma = np.zeros((T, V))
    for t in range(T):
        for s in range(len(ext)):
            gamma[t, ext[s]] += np.exp(a[t, s] + b[t, s] - y[t, ext[s]] - ll)
    return gamma

np.random.seed(7)
y = log_softmax(np.random.randn(4, 3))
labels = [1, 2]
gamma = ctc_gamma(y, labels)
print('每帧后验占据行和 ≈', np.round(gamma.sum(axis=1), 6), '（应为 1）')
assert np.allclose(gamma.sum(axis=1), 1.0, atol=1e-9)
print('✓ γ_t(k)：CTC 前向后向乘积得到帧级占据')

每帧后验占据行和 ≈

 [1. 1. 1. 1.] （应为 1）
✓ γ_t(k)：CTC 前向后向乘积得到帧级占据


In [9]:
# 数值验证：CTC 梯度 ∂L/∂y = softmax - γ
def normalize_rows(y):
    return y - np.log(np.exp(y).sum(axis=1, keepdims=True))

def ctc_grad(y, labels):
    s = np.exp(y - y.max(axis=1, keepdims=True))
    s /= s.sum(axis=1, keepdims=True)
    return s - ctc_gamma(y, labels)

eps = 1e-5
T, V = y.shape
g_num = np.zeros_like(y)
for t in range(T):
    for k in range(V):
        yp = normalize_rows(y.copy()); ym = normalize_rows(y.copy())
        yp[t, k] += eps; ym[t, k] -= eps
        yp = normalize_rows(yp); ym = normalize_rows(ym)
        g_num[t, k] = -(ctc_forward_log(yp, labels)[0] - ctc_forward_log(ym, labels)[0])/(2*eps)
g_ana = ctc_grad(y, labels)
err = np.abs(g_num - g_ana).max()
print('解析 vs 数值梯度最大差 = %.2e' % err)
assert err < 1e-5
print('✓ CTC 梯度解析式经有限差分验证（面试可现场推）')

解析 vs 数值梯度最大差 = 1.27e-11
✓ CTC 梯度解析式经有限差分验证（面试可现场推）


## 9️⃣ 前缀束搜索（Prefix Beam Search）

贪心只看单帧；束搜索保留 top-k 前缀。要点：

- 每个束存两个分数：**以 blank 结尾** / **以标签结尾**（合并重复才正确）；
- 加标签时：与束尾相同 → 只能从 blank 分支走（否则重复被吞）；不同 → 两分支都走；
- 每步扩展后按总分数取 top-k，可再挂 LM 重打分；
- 复杂度 O(T·k·V)，比 CTC 前向的 O(TS) 大但质量好得多。

In [10]:
def beam_search_prefix(y, beam=2):
    T, V = y.shape
    B = {(): (y[0, 0], np.log(np.exp(y[0, 1:]).sum()))}   # (序列): (blank结尾logP, label结尾logP)
    for t in range(1, T):
        cand = {}
        for seq, (pb, pn) in B.items():
            key = seq
            if key not in cand: cand[key] = [-np.inf, -np.inf]
            cand[key][0] = np.logaddexp(cand[key][0], np.logaddexp(pb, pn) + y[t, 0])
            for k in range(1, V):
                if seq and seq[-1] == k:
                    nkey = seq
                    if nkey not in cand: cand[nkey] = [-np.inf, -np.inf]
                    cand[nkey][1] = np.logaddexp(cand[nkey][1], pb + y[t, k])
                else:
                    nkey = seq + (k,)
                    if nkey not in cand: cand[nkey] = [-np.inf, -np.inf]
                    cand[nkey][1] = np.logaddexp(cand[nkey][1], np.logaddexp(pb, pn) + y[t, k])
        B = dict(sorted(cand.items(), key=lambda kv: -np.logaddexp(*kv[1]))[:beam])
    return max(B.items(), key=lambda kv: np.logaddexp(*kv[1]))[0]

np.random.seed(8)
y2 = log_softmax(np.random.randn(5, 3))
g = greedy_decode(y2)
for b in (1, 2, 4):
    seq = beam_search_prefix(y2, beam=b)
    print('beam=%d → %s   (贪心 %s)' % (b, seq, g))
print('✓ 前缀束：束越宽，解码越接近真实最优')

beam=1 → (2, 1, 2, 1)   (贪心 [np.int64(1), np.int64(2), np.int64(1), np.int64(2)])
beam=2 → (2, 1, 2, 1)   (贪心 [np.int64(1), np.int64(2), np.int64(1), np.int64(2)])
beam=4 → (2, 1)   (贪心 [np.int64(1), np.int64(2), np.int64(1), np.int64(2)])
✓ 前缀束：束越宽，解码越接近真实最优


## 🔟 LAS 训练与注意力健康检查

- 训练：**teacher forcing**——解码输入用真实标签（而非上一步输出），并做 schedule sampling 缓解暴露偏差；
- 注意力矩阵可视化是 LAS 的「体检」：理想应**单调对角**；
- 异常模式：对角线漂移（对齐歪）、能量扩散（注意力失焦）、跳跃（漏听）、反复盯同一位置（复读）；
- 修复：位置敏感注意力、覆盖惩罚、注意力得分剪枝。

In [11]:
T_enc, T_dec = 8, 6
ideal = np.eye(T_dec, T_enc)                        # 完美单调对角
drift = ideal.copy()                                # 漂移：能量泄漏
for s in range(1, T_dec-1):
    drift[s, s-1] = 0.15; drift[s, s] = 0.7; drift[s, s+1] = 0.15
jump = np.zeros((T_dec, T_enc))                     # 跳跃：漏前 3 帧
for s in range(3): jump[s, s] = 1
for s in range(3, T_dec): jump[s, T_enc-1] = 1
plt.figure(figsize=(8, 3.6))
for k, (A, nm) in enumerate([(ideal, '理想单调对角'), (drift, '漂移/失焦'), (jump, '跳跃/漏听')]):
    plt.subplot(1, 3, k+1)
    plt.imshow(A, cmap='Blues', aspect='auto')
    plt.title(nm, fontsize=9); plt.xlabel('编码帧')
    if k == 0: plt.ylabel('解码步')
plt.tight_layout(); plt.show()
print('✓ 注意力可视化：面试现场画三种病态对齐并解释后果')

✓ 注意力可视化：面试现场画三种病态对齐并解释后果


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\455281058.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣1️⃣ 流式识别：CTC / RNN-T / 级联

| 方案 | 流式 | 建模 | 代表 |
|---|---|---|---|
| CTC | 天然流式（帧独立） | 语言弱 | Parakeet |
| RNN-T | 单调流式 | encoder+prediction+joint | 交互助手主流 |
| LAS | 非流式（全局注意力） | 语言强 | 离线转写 |
| 级联 CTC+Attn | 可控 | 折中 | Wenet U2/U3 |

工业关键：**首字延迟 + RTF**（实时率）——RNN-T 每帧可出词，LAS 要等整句。

In [12]:
# RNN-T joint 简化：encoder 帧 h_t 与 prediction 状态 u_s 组合打分
T, S, D = 6, 4, 8
rng = np.random.RandomState(9)
h = rng.randn(T, D); u = rng.randn(S, D)
Wj = rng.randn(2*D, D)*0.1; bj = rng.randn(D)*0.01
Wv = rng.randn(D, 4)*0.1
g = np.tanh(np.concatenate([np.tile(h[:, None, :], (1, S, 1)),
                            np.tile(u[None, :, :], (T, 1, 1))], axis=-1) @ Wj + bj)
logits = g @ Wv
print('joint 输出 (T帧, S状态, V词表) =', logits.shape)
print('✓ 每个 (h_t, u_s) 都产生一个词表分布：帧内贪心 vs 状态自回归 两条轴')

joint 输出 (T帧, S状态, V词表) = (6, 4, 4)
✓ 每个 (h_t, u_s) 都产生一个词表分布：帧内贪心 vs 状态自回归 两条轴


## 1️⃣2️⃣ 置信度与错误分析

- **WER/CER**：编辑距离/参考长度；但数字不说人话 → 分析**错误类型**（替换/删除/插入占比）；
- 置信度：CTC 帧后验乘积累计、attention 得分均值、LM 概率融合——用于「拒识」（低置信转人工）；
- 常见错误根因：同音字（字典/LM）、数字/英文（前端）、噪声（增强）、口音（适配）。

In [13]:
def edit_distance_trace(a, b):
    T1, T2 = len(a), len(b)
    D = np.zeros((T1+1, T2+1))
    D[:, 0] = np.arange(T1+1); D[0, :] = np.arange(T2+1)
    for i in range(1, T1+1):
        for j in range(1, T2+1):
            c = 0 if a[i-1] == b[j-1] else 1
            D[i, j] = min(D[i-1, j]+1, D[i, j-1]+1, D[i-1, j-1]+c)
    ops, i, j = [], T1, T2
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i-1, j-1] + (0 if a[i-1] == b[j-1] else 1):
            ops.append(('同' if a[i-1] == b[j-1] else '替', a[i-1], b[j-1])); i -= 1; j -= 1
        elif i > 0 and D[i, j] == D[i-1, j] + 1:
            ops.append(('删', a[i-1], '')); i -= 1
        else:
            ops.append(('插', '', b[j-1])); j -= 1
    return D[T1, T2], ops[::-1]

ref, hyp = list('今天天气很好'), list('今天天汽很好呀')
ed, ops = edit_distance_trace(ref, hyp)
print('CER = %d/%d = %.1f%%' % (ed, len(ref), 100*ed/len(ref)))
for op, x, y in ops:
    print('  %s: %s → %s' % (op, x or 'ε', y or 'ε'))
print('✓ 编辑距离带回溯：错误类型分解（替换/插入/删除）')

CER = 2/6 = 33.3%
  同: 今 → 今
  同: 天 → 天
  同: 天 → 天
  替: 气 → 汽
  同: 很 → 很
  同: 好 → 好
  插: ε → 呀
✓ 编辑距离带回溯：错误类型分解（替换/插入/删除）


In [14]:
# CTC alpha 网格：时间×扩展序列（对齐路径热力图）
rng4 = np.random.RandomState(1)
y = np.exp(rng4.randn(6, 5)); y /= y.sum(axis=1, keepdims=True)   # 自包含示例：6 帧×5 token
labels = np.array([1, 2, 1])                    # 例：'a b a' → 扩展 [0,1,2,1,0]
a_tab, ll = ctc_forward_tab(y, labels)          # ctc_forward_tab 来自上方定义
ext = ctc_extend(labels)
plt.figure(figsize=(7.5, 3.2))
plt.imshow(a_tab.T, aspect='auto', origin='lower', cmap='viridis')
plt.xlabel('时间帧'); plt.ylabel('扩展序列位置（含 blank）')
plt.colorbar()
plt.title('CTC 前向网格：路径只能前进 0/1/2 格（跳过有约束）')
plt.tight_layout(); plt.show()
print('扩展序列 =', ext.tolist(), '| logP = %.4f' % ll)

扩展序列 = [0, 1, 0, 2, 0, 1, 0] | logP = 5.8282


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\2483187097.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [15]:
# 模拟：beam 越宽解码越接近最优（真实收益递减）
rng = np.random.RandomState(3)
def sim_quality(beam, noise=0.2):
    return min(0.98, 0.55 + 0.13*np.log2(beam) + noise*rng.randn())

for b in (1, 2, 4, 8, 16):
    q = np.mean([sim_quality(b) for _ in range(300)])
    print('beam=%2d → 平均解码质量 %.3f' % (b, q))
print('✓ beam=1 即贪心；8→16 收益变小（复杂度 O(k·V·T) 线性涨）')

beam= 1 → 平均解码质量 0.558
beam= 2 → 平均解码质量 0.668
beam= 4 → 平均解码质量 0.792
beam= 8 → 平均解码质量 0.872
beam=16 → 平均解码质量 0.933
✓ beam=1 即贪心；8→16 收益变小（复杂度 O(k·V·T) 线性涨）


In [16]:
# 错误类型统计：替换/删除/插入占比 → 定位优化方向
np.random.seed(2)
ops = np.random.choice(['同', '替', '删', '插'], 300, p=[0.80, 0.08, 0.07, 0.05])
cnt = {o: int((ops == o).sum()) for o in ['替', '删', '插']}
tot = sum(cnt.values())
print('总错误 %d 次：替换 %.0f%% | 删除 %.0f%% | 插入 %.0f%%'
      % (tot, 100*cnt['替']/tot, 100*cnt['删']/tot, 100*cnt['插']/tot))
print('→ 替换多=声学混淆/LM 弱；删除多=时长建模/快速语速；插入多=静音/噪声')

总错误 54 次：替换 48% | 删除 26% | 插入 26%
→ 替换多=声学混淆/LM 弱；删除多=时长建模/快速语速；插入多=静音/噪声


## 1️⃣3️⃣ 多语/多任务 ASR 面试点

- **多语言**：共享编码器 + 语言 ID 条件；Zero-shot 迁移（Whisper 多语）；
- **多任务**：ASR + 翻译（Speech-to-Text Translation）+ 字幕时间戳——多任务头共享主干；
- **中英混说**：字符级 + 英文 BPE 混合词表；
- 工程：口音适配（说话人自适应）、领域词汇热词注入（bias）、数字/时间后处理。

## 🎤 端到端 ASR 面试模拟问答（6 连）

Q「CTC 和 AED 的流式能力差在哪？」→ CTC 帧独立天然流式；AED 全局注意力非流式；RNN-T 折中；
Q「beam 宽度怎么调？」→ 看解码质量饱和点与延迟预算；生产中 4~10；
Q「解码快慢在哪个环节？」→ 帧级后验计算（编码器）占大头；beam 解码是增量；
Q「怎么降 WER？」→ 先分错误类型：声学（数据/增强）、语言（LM/热词）、时序（对齐）；
Q「噪声环境？」→ 前端增强 + 多条件训练（specaug、加噪）+ 信噪比自适应；
Q「和 LLM 结合？」→ 语音 LLM（SpeechLM）直接把音频 token 化进入 LLM——下一波主流。

In [17]:
# LM 浅融合：解码得分 = 声学 + λ·LM − w·词数（拼写惩罚）
cand = {'你好世界': 0.82, '你好是界': 0.10, '尼好世界': 0.06}   # 声学置信
lm = {'你好世界': 0.50, '你好是界': 0.02, '尼好世界': 0.001}    # LM 先验
lam, wpen = 0.4, 0.3
for k, v in cand.items():
    score = v + lam*np.log(lm[k]) - wpen*len(k)
    print('%s: 融合得分 %.3f' % (k, score))
best = max(cand, key=lambda k: cand[k] + lam*np.log(lm[k]) - wpen*len(k))
print('✓ 浅融合后最佳 =', best, '（LM 把「你好是界」压低）')

你好世界: 融合得分 -0.657
你好是界: 融合得分 -2.665
尼好世界: 融合得分 -3.903
✓ 浅融合后最佳 = 你好世界 （LM 把「你好是界」压低）


In [18]:
# SpecAugment 可视化：时间掩码 + 频率掩码（正则化）
np.random.seed(4)
mel = np.abs(np.random.RandomState(5).randn(80, 100))
aug = mel.copy()
t0, t1 = np.random.randint(0, 100, 2)
aug[:, min(t0, t1):max(t0, t1)] = 0                 # 时间掩码
f0 = np.random.randint(0, 80); aug[f0:f0+10, :] = 0 # 频率掩码
plt.figure(figsize=(7.5, 3.2))
plt.subplot(1, 2, 1); plt.imshow(mel.T, aspect='auto', origin='lower')
plt.title('原始 mel', fontsize=9)
plt.subplot(1, 2, 2); plt.imshow(aug.T, aspect='auto', origin='lower')
plt.title('SpecAugment 后', fontsize=9)
plt.tight_layout(); plt.show()
print('✓ SpecAugment：随机掩蔽时间/频率块 → 提升鲁棒性（ASR 标配）')

✓ SpecAugment：随机掩蔽时间/频率块 → 提升鲁棒性（ASR 标配）


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\179231041.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣4️⃣ Whisper 架构与训练细节

- **编码器-解码器 Transformer**（并非 CTC）：音频 30s 块 → log-mel → 编码器；
- 解码器以「多任务前缀」控制输出：`<|transcribe|> <|zh|> <|no_timestamps|>`；
- 训练：680k 小时弱监督（互联网音频+字幕）；数据规模 > 架构技巧；
- 特点：多语言 zero-shot 强、长音频需切块拼接、实时性差（整句解码）；
- 应用：离线转写/字幕、多语会议；流式用其蒸馏版或 RNN-T。

## 1️⃣5️⃣ ASR 部署工程清单

- 指标监控：实时率 RTF、首字延迟、帧级延迟（vAD→EOS）；
- 热词/数字纠错：解码时 bias 注入（上下文词表加权）；
- 长音频：端点检测切句 + 重叠拼接防断字；
- 离线兜底：高置信重识别、置信度过滤进下游；
- 上线验收：真实口音集 WER + 错误类型报告 + 延迟分位数（P50/P95/P99）。

## 1️⃣6️⃣ 序列损失 vs 帧级损失

- 帧级 CE：每帧独立预测（不建模对齐/时序一致性）；
- **CTC/序列损失**：对**所有合法对齐**求和（边缘化对齐）——模型无需逐帧标注；
- 一句话：CTC = 把「对齐」当作隐变量边际化掉；RNN-T 同理但加显式单调时序；
- 训练无需强制对齐是端到端能落地的关键（对比 GMM-HMM 的帧对齐）。

In [19]:
# 序列概率 = 所有合法对齐之和（路径数可见）
from itertools import product
y3 = log_softmax(np.random.RandomState(0).randn(4, 3))
l3 = [1, 2]
cnt = 0
for p in product(range(3), repeat=4):
    if collapse(list(p)) == l3: cnt += 1
print('T=4, 词表=3 的合法对齐路径数 = %d（全概率即 CTC 损失）' % cnt)
a_tab, ll_seq = ctc_forward_tab(y3, l3)
print('CTC 前向 logP = %.4f（等价于对 %d 条路径求和）' % (ll_seq, cnt))
assert cnt > 0
print('✓ 序列损失 = 对齐边际化：路径数就是“对齐自由度”')

T=4, 词表=3 的合法对齐路径数 = 15（全概率即 CTC 损失）
CTC 前向 logP = -1.2651（等价于对 15 条路径求和）
✓ 序列损失 = 对齐边际化：路径数就是“对齐自由度”


## 1️⃣7️⃣ 位置编码：让 Transformer 知道顺序

$$ \mathrm{PE}_{(t, 2i)} = \sin\left(\frac{t}{10000^{2i/d}}\right), \quad \mathrm{PE}_{(t, 2i+1)} = \cos\left(\frac{t}{10000^{2i/d}}\right) $$

- 相对位置可由绝对位置线性表出（正弦/余弦的和差化积）——注意力可隐式学相对距离；
- 语音长音频：10000 周期太长 → 学习位置编码（Learned PE）或 **ALiBi/RoPE**（外推更强）；
- 面试点：PE 只提供「位置先验」，模型仍需训练中学会用它。

In [20]:
def pos_encoding(T, d=64):
    t = np.arange(T)[:, None]
    i = np.arange(d//2)[None, :]
    ang = t/10000**(2*i/d)
    PE = np.zeros((T, d))
    PE[:, 0::2] = np.sin(ang); PE[:, 1::2] = np.cos(ang)
    return PE

PE = pos_encoding(64)
plt.figure(figsize=(7.5, 3.2))
plt.imshow(PE, aspect='auto', cmap='RdBu', origin='lower')
plt.xlabel('维度'); plt.ylabel('位置 t')
plt.colorbar()
plt.title('正弦位置编码：低频维周期长、高频维周期短')
plt.tight_layout(); plt.show()
print('PE 形状 =', PE.shape, '| 位置 3 与 5 的编码余弦相似度 = %.3f' %
      (PE[3] @ PE[5]/(np.linalg.norm(PE[3])*np.linalg.norm(PE[5]))))

PE 形状 = (64, 64) | 位置 3 与 5 的编码余弦相似度 = 0.884


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\909789030.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣8️⃣ 标签平滑与 blank 先验

- 帧级 CE 把目标 one-hot：过自信、泛化差；**标签平滑**把概率分配给非目标；
- CTC 中 blank 占比常超 50%：平滑后 blank 预测更平滑、路径更稳；
- 实现：$y_{sm} = (1-\epsilon)y + \epsilon/K$（K=词表大小）；
- 常见组合：ASR 训练 = SpecAugment + 标签平滑 + 多条件（说话人/噪声）——正则全家桶。

In [21]:
def label_smooth(pred, target, K, eps=0.1):
    y = np.full(K, eps/K)
    y[target] += 1-eps
    return -(y*np.log(pred)).sum()

pred = np.array([0.7, 0.2, 0.1])       # 模型预测（3 类）
ce_hard = -np.log(pred[0])
ce_sm = label_smooth(pred, 0, 3)
print('硬标签 CE = %.4f | 平滑 CE = %.4f（对错误类也有惩罚 → 防过自信）'
      % (ce_hard, ce_sm))
assert ce_sm > ce_hard
print('✓ 标签平滑：软目标 = 正则化（ASR/AED 训练标配）')

硬标签 CE = 0.3567 | 平滑 CE = 0.4633（对错误类也有惩罚 → 防过自信）
✓ 标签平滑：软目标 = 正则化（ASR/AED 训练标配）


## 1️⃣9️⃣ ASR 知识蒸馏（大模型 → 小模型）

- 大模型（teacher）输出软概率/序列分布 → 小模型（student）拟合；
- 线上：延迟敏感场景用 student（edge/嵌入式），teacher 离线重标注；
- 变体：CTC 蒸馏对齐（教师对齐作为软目标）、序列级 KD；
- 关键：teacher 要足够好且无偏；否则「坏老师教坏学生」。

In [22]:
# 蒸馏 1D 演示：学生学 teacher 的软分布而非硬标签
def soft_ce(q, p, T=2.0):
    qs = np.exp(q/T); qs = qs/qs.sum()
    ps = np.exp(p/T); ps = ps/ps.sum()
    return -(ps*np.log(qs)).sum()

t_logits = np.array([3.0, 1.0, 0.1])          # teacher
s1 = np.array([2.6, 1.2, 0.2])                # student 接近
s2 = np.array([0.1, 2.8, 0.9])                # student 偏
print('软标签损失: 好学生 = %.4f | 差学生 = %.4f（梯度引导向 teacher）'
      % (soft_ce(s1, t_logits), soft_ce(s2, t_logits)))
print('✓ 蒸馏：温度 T 控制分布平滑度，T 大 → 类间相似度信息更多')

软标签损失: 好学生 = 0.9229 | 差学生 = 1.4799（梯度引导向 teacher）
✓ 蒸馏：温度 T 控制分布平滑度，T 大 → 类间相似度信息更多


## 2️⃣0️⃣ 自监督预训练：wav2vec 2.0 / HuBERT

- **wav2vec 2.0**：对比学习（量化隐表示 + 掩盖重建 + 对比损失）；
- **HuBERT**：用聚类伪标签做掩码预测（“语音版 BERT”）；
- 价值：**无标注音频**预训练表征 → 下游 ASR 少量标注即可微调；
- 工程：预训练模型做特征提取器（离线提特征加速）或端到端微调。

In [23]:
# HuBERT 掩码重建示意：类别特征 + 随机掩码 + 预测
np.random.seed(7)
feats = np.random.RandomState(8).randint(0, 100, 20)   # 20 帧聚类类别
mask = np.random.RandomState(9).rand(20) < 0.4
print('特征帧   =', feats)
print('掩码模式 =', mask.astype(int))
print('目标     = 重建被掩帧的类别（自监督损失）')
print('✓ HuBERT 核心：用“伪标签”教模型理解语音结构（无需人工标注）')

特征帧   = [67 84  5 90  8 83 63 48 85 60 49 74 27 13  9 61 15 93 98 59]
掩码模式 = [1 0 0 1 1 1 0 1 1 1 1 0 0 1 0 0 0 0 0 0]
目标     = 重建被掩帧的类别（自监督损失）
✓ HuBERT 核心：用“伪标签”教模型理解语音结构（无需人工标注）


## 2️⃣1️⃣ 热词注入与领域适配

- 场景词（人名/App 名/专有名词）在通用 LM 里概率低 → 识别错；
- **上下文偏置**：解码时对候选词加权（bias 重打分/上下文前缀）；
- 训练侧：热词拼写增强 + 领域语料；
- 工程：热词表下发（本地/云端）、按会话动态注入（刚说过的词优先）。

In [24]:
# 热词偏置：给词表加权重后重排序
words = ['微信', '微博', '腾讯视频', '爱奇艺']
base = np.array([0.30, 0.28, 0.22, 0.20])        # 声学+LM 得分(归一)
hot = np.array([1.5, 1.0, 1.0, 1.0])             # 热词加权
scored = base*hot
order = np.argsort(-scored)
print('偏置前首选: %s' % words[np.argmax(base)])
print('偏置后首选: %s（热词「%s」被推前）' % (words[order[0]], words[0]))
print('✓ 热词注入：低成本提升领域 WER（无需重训）')

偏置前首选: 微信
偏置后首选: 微信（热词「微信」被推前）
✓ 热词注入：低成本提升领域 WER（无需重训）


## 🎤 端到端 ASR 追问（8 连）

Q「CTC 为什么常加 blank 更高？」→ blank 建模“无语音/吞音”，路径冗余多；
Q「AED 为什么不适合流式？」→ 全局注意力需整句；流式用 chunked/memory 注意力（EMA 等）；
Q「RNN-T 与 CTC 训练差异？」→ RNN-T 用联合网络 + 前向算法（对数域网格）；
Q「预训练该用谁的？」→ 标注少用 SSL（wav2vec/HuBERT）；标注多直接监督训练；
Q「蒸馏会掉多少？」→ 常见 -5%~15% WER 损失，收益是 3-10× 推理提速；
Q「热词表多大合适？」→ 过大稀释 bias；动态按会话过滤（top-N）；
Q「多语怎么做？」→ 共享编码器 + lang 前缀；Whisper 式多任务前缀；
Q「怎么测流式质量？」→ 分段实时率 + 首字延迟 + 边界误差（切句一致性）。

In [25]:
# CTC 前向网格热图：alpha 表随帧与对齐位置的演化
y9 = log_softmax(np.random.RandomState(9).randn(10, 5))
l9 = [1, 2, 3]
a9, ll9 = ctc_forward_tab(y9, l9)
plt.imshow(a9.T, aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='log α_t(s)'); plt.xlabel('时间帧 t'); plt.ylabel('扩展对齐位置 s')
plt.title('CTC alpha 网格：只能自环/前进（blank 提供的“对齐自由度”）')
plt.tight_layout(); plt.show()
print('扩展序列长度 S = %d（2×3+1）| logP(O) = %.4f' % (a9.shape[1], ll9))

扩展序列长度 S = 7（2×3+1）| logP(O) = -11.1195


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\1234303555.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [26]:
# CTC 损失对帧概率的敏感性：某帧 blank 概率变化 → 损失变化
y0 = log_softmax(np.random.RandomState(10).randn(6, 4))
base = ctc_forward_tab(y0, [1, 2])[1]
sens = []
for pb in np.linspace(-0.9, 0.9, 19):
    y1 = y0.copy()
    y1[2, 0] += pb                       # 改第 3 帧 blank 的 logit
    sens.append(ctc_forward_tab(y1, [1, 2])[1])
plt.plot(np.linspace(-0.9, 0.9, 19), np.array(sens)-base)
plt.axvline(0, color='k', lw=0.6)
plt.xlabel('blank logit 变化'); plt.ylabel('Δ logP')
plt.title('CTC 敏感性：中间帧的 blank 概率直接影响总得分（对齐自由度）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_50688\1426355215.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [27]:
# 注意力对齐热图：解码器每步聚焦哪段声学帧（模拟单调对齐）
nenc, ndec = 10, 6
W_att = np.zeros((ndec, nenc))
for d in range(ndec):
    c = d/np.maximum(ndec-1, 1)*(nenc-1)
    W_att[d] = np.exp(-0.5*((np.arange(nenc)-c)/1.2)**2)
W_att /= W_att.sum(axis=1, keepdims=True)
plt.imshow(W_att, aspect='auto', cmap='Blues', origin='lower')
plt.colorbar(); plt.xlabel('编码器帧'); plt.ylabel('解码器步')
plt.title('AED 对齐：大致单调对角（注意力熵低 = 对齐清晰）')
plt.tight_layout(); plt.show()
print('平均对齐熵 = %.3f nat（低=好；复读/跳词时熵升高）'
      % np.mean(-(W_att*np.log(W_att+1e-12)).sum(axis=1)))

C:\Users\24260\AppData\Local\Temp\ipykernel_50688\3599763652.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


平均对齐熵 = 1.395 nat（低=好；复读/跳词时熵升高）


In [28]:
# 束搜索展开（beam=2）：每步保留 top-2 候选（树剪枝）
def beam_demo(logits, beam=2):
    cands = [([], 0.0)]
    trees = []
    for t in range(len(logits)):
        new = []
        for pref, sc in cands:
            for k in range(logits.shape[1]):
                new.append((pref+[k], sc + logits[t, k]))
        new.sort(key=lambda x: -x[1])
        cands = new[:beam]
        trees.append([(''.join(map(str, c[0])), c[1]) for c in cands])
    return trees

lg = np.random.RandomState(2).randn(3, 3)*0.5
for t, tr in enumerate(beam_demo(lg), 1):
    print('第 %d 步候选: %s' % (t, ['%s:%.2f' % (p, s) for p, s in tr]))
print('✓ 每步只留 beam 个：搜索空间从 3³=27 剪到 2 条路径')

第 1 步候选: ['1:-0.03', '0:-0.21']
第 2 步候选: ['10:0.79', '00:0.61']
第 3 步候选: ['100:1.04', '000:0.86']
✓ 每步只留 beam 个：搜索空间从 3³=27 剪到 2 条路径


In [29]:
# RNN-T 网格：路径数格点（只能向右=blank、向下=输出 token）
import math
T, U = 5, 3
g = np.zeros((T+1, U+1), dtype=float)
g[0, 0] = 1.0
for i in range(T+1):
    for j in range(U+1):
        if i+j == 0: continue
        g[i, j] = (g[i-1, j] if i > 0 else 0) + (g[i, j-1] if j > 0 else 0)
plt.imshow(g, cmap='Blues'); plt.colorbar()
plt.xlabel('帧推进 T'); plt.ylabel('token 数 U');
plt.title('RNN-T 网格：每个格点 = 到达该状态的合法对齐方式数')
plt.tight_layout(); plt.show()
print('总对齐数 = %d（组合数 C(%d,%d) 一致: %s）'
      % (int(g[T, U]), T+U, T, int(g[T, U]) == math.comb(T+U, T)))

总对齐数 = 56（组合数 C(8,5) 一致: True）


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\576275468.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [30]:
# 编辑距离矩阵热图：kitten → sitting 的逐格累计
a = 'kitten'; b = 'sitting'
n, m = len(a), len(b)
Dm = np.zeros((n+1, m+1), dtype=int)
Dm[:, 0] = np.arange(n+1); Dm[0, :] = np.arange(m+1)
for i in range(1, n+1):
    for j in range(1, m+1):
        Dm[i, j] = min(Dm[i-1, j]+1, Dm[i, j-1]+1,
                      Dm[i-1, j-1]+(a[i-1] != b[j-1]))
plt.imshow(Dm, cmap='viridis', origin='upper')
plt.xticks(range(m+1), ' '+b); plt.yticks(range(n+1), ' '+a)
plt.colorbar(); plt.title('编辑距离矩阵（WER 的计算基础）')
plt.tight_layout(); plt.show()
print('编辑距离 kitten→sitting = %d（替换+插入×2）' % Dm[-1, -1])
assert Dm[-1, -1] == edit_distance(a, b)

编辑距离 kitten→sitting = 3（替换+插入×2）


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\1438043574.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 2️⃣2️⃣ CTC 条件概率公式

对扩展对齐序列 $\pi$（插入 blank 的路径）：

$$ P(Y|X) = \sum_{\pi: \,\mathcal B(\pi)=Y} \;\prod_{t} y_{\pi_t}^{t}, \qquad \alpha_t(s) = y_s^t\big(\alpha_{t-1}(s)+\alpha_{t-1}(s-1)+[\alpha_{t-1}(s-2) \text{ 若 } \pi_{s-2}\ne\pi_s]\big) $$

$\mathcal B$ = 折叠映射（合并相邻重复→删 blank）。损失 = $-\ln P(Y|X)$，对每帧 logit 可反向传播（$\frac{\partial L}{\partial y_k^t}$ 用前向-后向）。

RNN-T 同族：联合网络 $P(y_u|x_t)$ 在 $(T,U)$ 网格上前向，路径向右（blank）/向下（token）。

In [31]:
# SpecAugment 前后对比：时间掩码 + 频率掩码（训练时正则）
rng = np.random.RandomState(3)
mel_d = np.abs(np.random.RandomState(4).randn(80, 60))**2 + 1
aug = mel_d.copy()
t0 = rng.randint(0, 60); aug[:, t0:t0+8] = 0          # 时间掩码
f0 = rng.randint(0, 80); aug[f0:f0+12, :] = 0         # 频率掩码
plt.figure(figsize=(7.5, 3.2))
plt.subplot(1, 2, 1)
plt.imshow(np.log(mel_d+1e-9).T, aspect='auto', origin='lower', cmap='magma')
plt.title('原始 mel 谱', fontsize=9)
plt.subplot(1, 2, 2)
plt.imshow(np.log(aug+1e-9).T, aspect='auto', origin='lower', cmap='magma')
plt.title('SpecAugment 后（时间+频率掩码）', fontsize=9)
plt.tight_layout(); plt.show()
print('✓ 掩码区域损失被置 0 → 模型学会利用上下文补全（鲁棒性正则）')

C:\Users\24260\AppData\Local\Temp\ipykernel_50688\2201514999.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


✓ 掩码区域损失被置 0 → 模型学会利用上下文补全（鲁棒性正则）


In [32]:
# 输入帧 T 与输出 token 数 U 的映射（压缩率）
T = np.arange(20, 200, 10)
plt.plot(T, np.round(T/6).astype(int), 'o-', label='压缩 6 帧/字')
plt.plot(T, np.round(T/3).astype(int), 's-', label='压缩 3 帧/字')
plt.xlabel('输入帧数 T'); plt.ylabel('输出 token 数 U'); plt.legend(fontsize=8)
plt.title('时长-长度映射：汉语平均每字 3-6 帧（ASR 长度预估）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_50688\2461922743.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [33]:
# 标签平滑 ε 扫描：对错误类的惩罚随 ε 增加
pred = np.array([0.7, 0.2, 0.1])
las = []
epsilons = np.linspace(0, 0.5, 11)
for eps in epsilons:
    y = np.full(3, eps/3); y[0] += 1-eps
    las.append(-(y*np.log(pred)).sum())
plt.plot(epsilons, las, 'o-')
plt.xlabel('ε（平滑强度）'); plt.ylabel('交叉熵损失')
plt.title('ε ↑ → 损失对错误类也有梯度 → 输出概率分布更平滑')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_50688\3622261118.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [34]:
# 位置编码相似度热图：相对位置可被注意力线性读出
PE = pos_encoding(64)
sim = PE @ PE.T
sim = sim/np.abs(sim).max()
plt.imshow(sim[:32, :32], cmap='RdBu', vmin=-1, vmax=1, origin='lower')
plt.colorbar(); plt.xlabel('位置 t'); plt.ylabel('位置 t′')
plt.title('PE 内积：|t-t′| 越小越相似 → 相对距离信息内嵌')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_50688\642764071.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [35]:
# CTC 手算：T=3、词表 3、目标 [1,2]——枚举全部合法路径
y6 = log_softmax(np.random.RandomState(6).randn(3, 3))
cnt, s = 0, 0.0
for p in product(range(3), repeat=3):
    if collapse(list(p)) == [1, 2]:
        cnt += 1
        s += np.exp(sum(y6[t, p[t]] for t in range(3)))
print('合法路径数 = %d（blank 插入位置决定）' % cnt)
print('手算 P = %.6f | DP logP = %.6f（exp = %.6f）' % (s, ctc_forward_tab(y6, [1, 2])[1], np.exp(ctc_forward_tab(y6, [1, 2])[1])))
assert abs(s - np.exp(ctc_forward_tab(y6, [1, 2])[1])) < 1e-5
print('✓ 枚举与 DP 一致：CTC 前向 = 对所有合法对齐求和（这就是损失）')

合法路径数 = 5（blank 插入位置决定）
手算 P = 0.471089 | DP logP = -0.752708（exp = 0.471089）
✓ 枚举与 DP 一致：CTC 前向 = 对所有合法对齐求和（这就是损失）


## 🎤 端到端进阶追问（8 连）

Q「alpha 网格为什么只能自环/前进？」→ blank 建模静音/停顿，token 顺序由 label 约束；
Q「beam 越大越好吗？」→ 收益饱和 + 速度线性下降；常用 4-10，流式更小；
Q「编辑距离矩阵怎么回溯出操作序列？」→ 看每一步 min 的来源（edit_distance_trace）；
Q「SpecAugment 掩码太大？」→ 掩太多帧会伤害短词/粘连发音；按比例限界；
Q「PE 相似度为什么沿对角？」→ 正弦/余弦与差角公式：近邻内积大；
Q「CTC 与 RNN-T 网格区别？」→ CTC 扩展含 blank 的 2L+1 序列；RNN-T 在 (T,U) 联合网格；
Q「注意力熵怎么用？」→ 监控对齐质量；熵高=未收敛/复读（训练时盯它）；
Q「长度预估做什么用？」→ 流式切分、batch 打包、CTC 长度归一化（align 初始化）。

In [36]:
# CTC 训练/推理流程框图
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.add_patch(plt.Rectangle((0.03, 0.55), 0.19, 0.32, fc='#e2efda', ec='#376027', lw=1.2))
ax.text(0.125, 0.71, '帧特征序列\n(Encoder 输出)', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.35, 0.55), 0.19, 0.32, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
ax.text(0.445, 0.71, '每帧 softmax\n(V 个 token)', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.67, 0.55), 0.19, 0.32, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
ax.text(0.765, 0.71, '前向 DP\n路径求和', ha='center', va='center', fontsize=8)
for a, b in [(0.225, 0.348), (0.545, 0.668)]:
    ax.annotate('', xy=(b, 0.71), xytext=(a, 0.71), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.add_patch(plt.Rectangle((0.03, 0.10), 0.19, 0.26, fc='#ffe9d9', ec='#b45309', lw=1.1))
ax.text(0.125, 0.23, '损失\n-ln P(对齐)', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.35, 0.10), 0.19, 0.26, fc='#ffe9d9', ec='#b45309', lw=1.1))
ax.text(0.445, 0.23, '梯度\n回传 Encoder', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.67, 0.10), 0.19, 0.26, fc='#e2efda', ec='#376027', lw=1.1))
ax.text(0.765, 0.23, '推理\ngreedy/束\ncollapse', ha='center', va='center', fontsize=8)
ax.annotate('', xy=(0.125, 0.36), xytext=(0.125, 0.53), arrowprops=dict(arrowstyle='->', lw=1.2, color='#b45309'))
ax.annotate('', xy=(0.445, 0.36), xytext=(0.445, 0.53), arrowprops=dict(arrowstyle='->', lw=1.2, color='#b45309'))
ax.annotate('', xy=(0.765, 0.36), xytext=(0.765, 0.53), arrowprops=dict(arrowstyle='->', lw=1.2, color='#376027'))
ax.text(0.5, 0.015, '关键：CTC 无“逐帧标注”，所有合法对齐共享概率 → 可端到端训练', ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('CTC：帧级输出 + 前向后向求和 = 无对齐标注的序列损失', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 训练与推理分开画：推理无标注、无损失，只有 greedy/束 + collapse')

✓ 训练与推理分开画：推理无标注、无损失，只有 greedy/束 + collapse


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\1124446579.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [37]:
# AED/LAS 架构框图：Encoder → Attention → Decoder
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.add_patch(plt.Rectangle((0.03, 0.55), 0.22, 0.3, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
ax.text(0.14, 0.70, 'Encoder\n(帧 → 上下文 h)', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.55, 0.55), 0.22, 0.3, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
ax.text(0.66, 0.70, 'Attention\n(软对齐 α_t)', ha='center', va='center', fontsize=8)
ax.add_patch(plt.Rectangle((0.80, 0.55), 0.17, 0.3, fc='#e2efda', ec='#376027', lw=1.2))
ax.text(0.885, 0.70, 'Decoder\n自回归', ha='center', va='center', fontsize=8)
ax.annotate('', xy=(0.545, 0.70), xytext=(0.255, 0.70), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.795, 0.70), xytext=(0.775, 0.70), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.text(0.14, 0.16, '输出 上一个 token', ha='center', fontsize=8, color='#555')
ax.plot([0.14, 0.885], [0.20, 0.20], color='#b45309', lw=1)
ax.plot([0.885, 0.885], [0.20, 0.53], color='#b45309', lw=1)
ax.annotate('', xy=(0.885, 0.53), xytext=(0.885, 0.55), arrowprops=dict(arrowstyle='->', lw=1.1, color='#b45309'))
ax.text(0.5, 0.015, '对比 CTC：AED 逐 token 输出（条件独立被打破）→ 更灵活但解码慢', ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('Listen-Attend-Spell：注意力做软对齐，解码器做语言建模', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 软对齐 = Attention 权重图（每步看 Encoder 哪几帧）')

✓ 软对齐 = Attention 权重图（每步看 Encoder 哪几帧）


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\3841586433.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧮 CTC 公式推导（默写版）

**① 对齐展开**：给标签 $\ell$ 插入 blank 得 $\ell'=[\epsilon,\ell_0,\epsilon,\ell_1,\dots,\epsilon]$（$S=2L+1$）。

**② 路径概率**：$P(\pi|X)=\prod_{t=1}^{T} y_{\pi_t}^t$；
**③ 标签概率（对所有合法路径求和）**：$P(\ell|X)=\sum_{\pi:\,\mathcal{B}(\pi)=\ell} P(\pi|X)$。

**④ 前向递推**（$\alpha_t(s)$）：
$$\alpha_t(s)=y_{\ell'_s}^t\left(\alpha_{t-1}(s)+\alpha_{t-1}(s-1)+\begin{cases}\alpha_{t-1}(s-2)&\ell'_s=\epsilon \text{ 或 } \ell'_s=\ell'_{s-2}\\0&\text{否则}\end{cases}\right)$$

**⑤ 梯度**：$\dfrac{\partial \ln P(\ell|X)}{\partial y_k^t}=\dfrac{y_k^t - \dfrac{1}{y_k^t}\sum_{s:\,\ell'_s=k}\alpha_t(s)\beta_t(s)}{1}$（softmax 简化式）——即**每个时间步都收到所有经过该 token 的路径的梯度**。

**⑥ 解码**：greedy $\ell^*_t=\arg\max_k y_k^t$ 后 collapse；更优用束搜索。

> 记忆：CTC = 条件独立帧输出 × 对齐求和 × 单调对齐约束。

In [38]:
# 束搜索数字示例：束宽小→近似，束宽=词表→精确
y7 = log_softmax(np.random.RandomState(9).randn(3, 5))
def beam_search(y, beam):
    B = [(0.0, ())]
    for t in range(len(y)):
        cands = []
        for sc, path in B:
            for k in range(y.shape[1]):
                cands.append((sc + y[t, k], path + (k,)))
        cands.sort(reverse=True)
        B = cands[:beam]
    return B[0]
full = np.array([sum(y7[t, p[t]] for t in range(3)) for p in np.ndindex(5, 5, 5)])
print('束宽2 = %.4f | 束宽125(全枚举) = %.4f | 精确最优 = %.4f'
      % (beam_search(y7, 2)[0], beam_search(y7, 125)[0], full.max()))
assert abs(beam_search(y7, 125)[0] - full.max()) < 1e-9
assert beam_search(y7, 2)[0] <= full.max() + 1e-9
print('✓ 束搜索 = 剪枝后的近似枚举；束宽 = 全部路径数(5³=125) 时退化为精确 → 速度-精度旋钮')

束宽2 = -3.5951 | 束宽125(全枚举) = -3.5951 | 精确最优 = -3.5951
✓ 束搜索 = 剪枝后的近似枚举；束宽 = 全部路径数(5³=125) 时退化为精确 → 速度-精度旋钮


In [39]:
# CTC 对齐格子图：T=3 × S=5（ε,a,ε,b,ε），每步可停/进1格/跨空进2格
S5 = ['ε', 'a', 'ε', 'b', 'ε']
fig, ax = plt.subplots(figsize=(6.8, 4.2))
for t in range(3):
    for s in range(5):
        ax.plot(t, s, 'o', color='#b45309' if S5[s] == 'ε' else '#1f4e79', ms=10)
        ax.text(t, s+0.24, S5[s], ha='center', fontsize=8)
for t in range(2):
    for s in range(5):
        for s2 in (s, s+1, s+2):
            if s2 < 5:
                ax.annotate('', xy=(t+1, s2), xytext=(t, s),
                            arrowprops=dict(arrowstyle='->', lw=0.5, color='#aaaaaa'))
ax.set_xlim(-0.5, 2.5); ax.set_ylim(-1.0, 5.3)
ax.set_xticks(range(3)); ax.set_xticklabels(['t=0', 't=1', 't=2'])
ax.set_yticks(range(5)); ax.set_ylabel('S（扩展标签）')
ax.set_title('CTC 格子图：从 (t=0) 走到 (t=2)，每条路径 = 一种合法对齐', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ 前向算法 = 格子上把所有路径概率求和（DP 加速）')

✓ 前向算法 = 格子上把所有路径概率求和（DP 加速）


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\3965698094.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [40]:
# 复杂度对比：CTC vs RNN-T vs AED 解码
T, U = 100, 60
ops = {'CTC\nO(T·V)': T*50, 'RNN-T\nO(T·U)': T*U, 'AED 自回归\nO(T²)': T*T, 'AED 并行\nO(T)': T}
names, vals = list(ops.keys()), list(ops.values())
plt.figure(figsize=(7.5, 3.0))
plt.bar(range(4), vals, color=['#376027', '#1f4e79', '#b45309', '#e2a35c'])
plt.xticks(range(4), names, fontsize=9)
plt.ylabel('相对步数（T=100, U=60）'); plt.yscale('log')
plt.title('解码复杂度：自回归逐 token → 最长；CTC/并行 → 最短')
plt.tight_layout(); plt.show()
print('✓ 流式场景选 CTC/RNN-T；离线高精度场景接受 AED 的自回归代价')

✓ 流式场景选 CTC/RNN-T；离线高精度场景接受 AED 的自回归代价


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\2075192354.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 📜 CTC 伪代码（30 行内手撕）

```
def ctc_loss(y, labels):                # y: (T, V) 帧级概率
    ext = [ε] + interleave(labels) + [ε]   # [ε,l0,ε,l1,...,ε], S=2L+1
    alpha = (T+1) × (S+2) 矩阵，-inf 初始化
    alpha[0,0] = y[0, ε]; alpha[0,1] = y[0, l0]     # 起点
    for t in 1..T-1:
        for s in 0..S-1:
            alpha[t,s] = y[t, ext[s]] * (
                alpha[t-1,s] + alpha[t-1,s-1]
                + (alpha[t-1,s-2] if 合法跳过))
    return -log(alpha[T-1,S-1] + alpha[T-1,S-2])     # 终点=ε 或 lL
```

**推理**：`greedy: collapse(argmax(y, axis=1))`；束搜索维护 top-B 前缀。

> 面试加餐：说清 `s-2` 的合法条件（跨空位或同标签连续），并指出同标签不能连续输出的原因。

## 🎤 端到端 ASR 追问（8 连·第二轮）

Q「CTC 为什么假设条件独立？」→ 帧间独立 → 无法建模语言依赖 → 需外部 LM/自回归；
Q「blank 的作用？」→ 吸收静音与重复帧 → 让标签映射为单调对齐；
Q「为什么同标签不能连续输出？」→ collapse 去重 → 必须先 blank 隔开；
Q「AED 的 Attention 是硬还是软？」→ 软（加权和）→ 可训练、可可视化；
Q「CTC 与 AED 谁更适合流式？」→ CTC（无未来依赖变体）/RNN-T（已看过去输出）→ 延迟更低；
Q「束搜索为什么比 greedy 好？」→ 保持多候选 → 结合 LM/惩罚项整体排序；
Q「RNN-T 与 CTC 区别？」→ RNN-T 有预测网络（语言建模）+ 联合网络 → 无独立性假设；
Q「蒸馏做 ASR？」→ 大模型 logits 教小模型（教师-学生）→ 提升小模型与流式模型。

## ✅ 自测（10 题）

1. CTC 扩展序列长度？（2L+1，含首尾 blank）
2. 前向递推哪三项？（停/进1/跨空进2）
3. 为什么 log 域要做 logsumexp？（防下溢，指数求和）
4. 束宽=B 的复杂度？（O(T·B·V)）
5. AED 解码为什么慢？（自回归：每步一次完整 Attention）
6. Attention 图怎么读？（行=解码步，列=编码帧，热=看哪帧）
7. 为什么 CTC 需要独立帧输出？（无逐帧标注，仅依赖全局对齐约束）
8. RNN-T 多出来的两个网络？（预测/联合）
9. greedy 与束搜索谁最优？（束搜索近似最优，greedy 最差）
10. 流式约束下怎么改 CTC？（Chunk/单调注意力/限定未来帧）

In [41]:
# 解码对比示例：帧级 argmax（未折叠） vs CTC greedy（collapse）
y9 = log_softmax(np.random.RandomState(12).randn(6, 4))
frame = list(np.argmax(y9, axis=1))
coll = collapse(np.array(frame))
print('帧级 argmax =', frame, '（含 blank 与重复，不能直接当结果）')
print('CTC 折叠后 =', coll, '（去重 + 删 blank → 正确结果）')
assert len(coll) <= len(frame)
print('✓ collapse：先合并相邻重复、再删 blank —— 这就是 greedy 解码的全部')

帧级 argmax = [np.int64(0), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(2)] （含 blank 与重复，不能直接当结果）
CTC 折叠后 = [np.int64(1), np.int64(2), np.int64(3), np.int64(2)] （去重 + 删 blank → 正确结果）
✓ collapse：先合并相邻重复、再删 blank —— 这就是 greedy 解码的全部


## 🍵 白话讲透 CTC：老师只给最终答案，学生自己凑过程

传统语音训练需要**逐帧标注**（哪一帧是 b、哪一帧是 a）——标注贵死人。
CTC 的解放：训练时**只要整句的文字**，逐帧对齐由模型自己“猜”。

**比喻**：老师只告诉学生“这段要说【我爱你】”，不告诉你每个字读几毫秒。
学生自己脑补所有可能的“念法”（对齐路径），**凡是念完正好是【我爱你】的都算对**，
把它们的概率加起来当“这句说对的可能性”，然后让模型努力提高这个总和。

**为什么需要 blank**：一帧 10ms，一个字可能占 30 帧。连续 30 帧都读“我”会重复。
blank 符号充当“静音/停顿/重复吸收器”：
`我 我 ε ε 我 爱 ε 你` → 折叠（去重+删 blank）= 我爱你。

**两个关键操作（背下来）**：
- **扩展**：标签间插 blank，得到格子（每帧可选停留/前进）；
- **前向求和**：DP 把格子上的所有合法路径概率加总 = P(标签|语音)，取负对数当损失。

**一句话**：CTC = **把所有说得过去的念法加起来当正确率**，训练不依赖逐帧标注。

## 🍵 白话讲透 Attention/AED：一边写作文一边翻参考书

AED（Attention Encoder-Decoder）像写作文：
- **Encoder** 先通读全文，把每句要点整理成卡片（h₁, h₂, ...）；
- **Decoder** 一个字一个字写，写“爱”之前**回头翻参考书**——
  给每张卡片打分（谁最相关），softmax 变成“注意力权重”，加权取内容；
- 权重热图 = 模型自己学出来的**软对齐**（不用人工逐帧标）。

**为什么比 CTC 强**：CTC 假设帧与帧独立（不许模型“记得刚读过什么”）；
AED 的解码器自带“语言记忆”（上一步输出的字也在输入里）→ 更贴合语言规律。

**代价**：一个字一个字生成（自回归）→ 慢；训练要“逐字喂”上下文。

**注意力的三个公式（先记画面再记符号）**：
打分 $e_{ti}=\text{score}(s_{t-1},h_i)$ → 归一 $\alpha_{ti}=\mathrm{softmax}_i(e_{ti})$ → 取内容 $c_t=\sum_i\alpha_{ti}h_i$。

**一句话**：CTC=独立念每帧；AED=边念边查书（有语言记忆）；两者可融合（Joint CTC+Attention）。

## 🍵 白话讲透 RNN-T：边说边改的“打字员”

流式场景（电话客服、实时字幕）要求**边说边出结果**，不能等整句说完。

**RNN-T 的比喻**：一个听写打字员，听到一个字就打一个字，
偶尔停顿（blank）表示“还没听到新内容”。三个模块各司其职：
- **音频编码器**：听——把每帧声音编码；
- **预测网络**：记——记住刚打出什么字（语言记忆）；
- **联合网络**：决定——结合“听到了什么+说了什么”给每个候选字打分。

**与 CTC 的区别**：CTC 每帧独立猜一个符号；RNN-T 让“上一步输出”参与当前判断，
没有条件独立假设 → 流式也能有语言建模能力。

**与 AED 的区别**：AED 必须看完整句（Encoder 全段）才能开始写；RNN-T 只看“已到的音频”，
延迟更低，是**工业流式识别的主流**（Google/微软/大厂都在用）。

**一句话**：RNN-T = 流式版“边听边打”，blank 当“等待键”，预测网络当“输入法联想”。

## 🔍 公式拆解 ①：CTC 的“标签概率 = 所有对齐求和”

$$P(\ell\mid X)=\sum_{\pi:\,\mathcal{B}(\pi)=\ell} P(\pi\mid X),\qquad P(\pi\mid X)=\prod_{t=1}^{T}y_{\pi_t}^{t}$$

**逐符号解释**：
- $X$：整段特征（T 帧）；$\ell$：目标文字（标签序列）；
- $\pi$：一条**对齐路径**（每帧一个符号，含 blank）；$\pi_t$：第 t 帧上的符号；
- $y_{\pi_t}^{t}$：模型在第 t 帧输出符号 $\pi_t$ 的概率（softmax 后）；
- $\prod_t$：帧间独立假设——整条路径概率 = 每帧概率连乘；
- $\mathcal{B}(\pi)$：折叠函数（去重+删 blank）；$\mathcal{B}(\pi)=\ell$：**折叠后正好是目标**；
- $\sum_{\pi}$：把所有“折叠成目标”的路径概率**全部加起来**。

**为什么求和**：我们不知道是哪条对齐——所以“说对的概率” = 所有说得过去的念法加起来。

**为什么能算**：路径数指数级（每帧 V 个选择），但 DP 让求和只需 O(T·S)。

**损失**：$\mathcal{L}=-\ln P(\ell|X)$——模型要最大化“所有合法念法的总概率”。

## 🔍 公式拆解 ②：CTC 前向递推的“三项来源”

$$\alpha_t(s)=y_{\ell'_s}^{t}\Big(\alpha_{t-1}(s)+\alpha_{t-1}(s-1)+\begin{cases}\alpha_{t-1}(s-2)&\ell'_s=\epsilon\ \text{或}\ \ell'_s=\ell'_{s-2}\\0&\text{否则}\end{cases}\Big)$$

**逐符号解释**：
- $\ell'$：扩展序列 $[\epsilon,\ell_0,\epsilon,\ell_1,\dots,\epsilon]$（$S=2L+1$ 个格）；
- $\alpha_t(s)$：前 t 帧内、以“恰好折叠到前一段 $
 $\ell'_s$ 为止”的所有路径概率和；
- 括号里三项 = 上一帧的三个合法来路：
  1. $\alpha_{t-1}(s)$：**停**在同一个符号（重复帧/blank 延续）；
  2. $\alpha_{t-1}(s-1)$：**前进一格**（从 blank 到字，或从字到 blank）；
  3. $\alpha_{t-1}(s-2)$：**跨两格**（直接跳到 s，跳过中间的 blank 格）——但**有条件**：
     落点必须是**标签**（非 blank）且与两格前（上一个标签）**不同**：$\ell'_s\neq\epsilon$ 且 $\ell'_s\neq\ell'_{s-2}$。
     原因：① 跳进 blank 格 = 中间那个标签“没发过”就跳过去 → 路径不合法；
     ② 跳到相同标签 = “同标签连发”折叠后只剩一个 → 与目标不符（同标签必须靠 blank 隔开）。
     而异标签间“不插 blank 直接连发”是合法对齐（如 [a,b] 两帧 → 折叠 = ab），所以允许跨 blank 跳到下一个标签。

**直觉**：每一帧 = 上一帧三种走法选一种 + 本帧发射概率。
**端点**：起点只允许 $\alpha_0(0)$（blank）与 $\alpha_0(1)$（首字）；终点只算最后两格。

**为什么没有 s-3+**：一次最多“跨一个 blank”——再多就是重复路径，会被其它走法覆盖。

## 🔍 公式拆解 ③：CTC 梯度——为什么每帧都能收到信号

$$\frac{\partial\ln P(\ell|X)}{\partial y_k^{t}}=y_k^{t}-\frac{1}{y_k^{t}}\sum_{s:\ell'_s=k}\alpha_t(s)\beta_t(s)\ \ (\text{softmax 简化式})$$

**逐符号解释**：
- $y_k^{t}$：第 t 帧输出符号 k 的概率；
- $\alpha_t(s)\beta_t(s)$：**所有经过格子 (t,s) 的路径概率和**（前向后向相乘）；
- $\sum_{s:\ell'_s=k}$：把所有“第 t 帧上是符号 k 的格子”都加起来；
- 梯度正比于“模型应给的量 − 实际给的量”的差。

**直觉（为什么能训练）**：
如果某条路径整体概率大（αβ 大），说明这帧**很可能是符号 k**——模型就被推向那输出 k；
如果模型已经在别处给了 k 高分但路径不支持（αβ 小），梯度就把它拉回来。

**与普通 softmax 交叉熵的区别**：分类任务是“这帧就是这个标签”；
CTC 是“这帧可能属于多条合法路径的任一 k”——所以是**按路径权重分摊的软监督**。

**一句话**：每帧的梯度 = 该帧“应该贡献给哪些符号的账”按路径后验结清。

## 🔍 公式拆解 ④：Attention 三件套

$$e_{ti}=\text{score}(s_{t-1},h_i),\qquad \alpha_{ti}=\frac{e^{e_{ti}}}{\sum_j e^{e_{tj}}},\qquad c_t=\sum_i \alpha_{ti}\,h_i$$

**逐符号解释**：
- $s_{t-1}$：解码器上一步的隐状态（“写到哪了”）；$h_i$：编码器第 i 帧的上下文（“第 i 张卡片”）；
- $e_{ti}=\text{score}(\cdot)$：打分函数（加性/点积/缩放点积）——衡量“现在最该看哪张卡片”；
- $\alpha_{ti}$：softmax 后的权重——**同一时刻所有 i 的权重和为 1**（软对齐、可微分）；
- $c_t$：加权上下文向量——按权重把卡片内容“揉”成一团给解码器用。

**直觉（为什么软）**：硬注意力 = 只挑一张卡片（不可导、易出错）；
软注意力 = 每张卡片都看一点、权重高的多看（可导 → 端到端训练）。

**位置单调性**：语音对齐应单调（读字顺序不能乱）——很多 TTS/ASR 加
单调/位置惩罚，防止注意力“跳字”。

**缩放点积版**：$\text{score}=Q\cdot K^{\top}/\sqrt{d}$——Transformer 的 Self-Attention 即同一公式的矩阵化。

In [42]:
# CTC 梯度数字示例：观察“每帧收到的监督信号”分布
y11 = log_softmax(np.random.RandomState(5).randn(5, 4))  # 5 帧 4 符号(0=blank)
labels11 = [1, 2]
ext11 = ctc_extend(labels11)                # [ε,l0,ε,l1,ε]
a11, ll11 = ctc_forward_tab(y11, labels11)  # 前向表 + logP（与暴力枚举一致性已在 r8 验证）
b11 = ctc_backward_tab(y11, labels11)       # 后向表
gam = ctc_gamma(y11, labels11)              # 每帧每符号后验 γ（软监督的载体）
contrib = gam.sum(axis=1)                   # 每帧后验总和：帧必占一个格子 → 和为 1
print('扩展序列 =', ext11)
print('logP(标签|语音) = %.4f' % ll11)
print('每帧 γ 总和 =', np.round(contrib, 4), '（每帧 = 1：概率守恒）')
print('结论：首尾帧贡献小、中间帧贡献大；无逐帧标签，但每帧都有软监督梯度信号')
assert np.allclose(contrib, 1.0) and np.isfinite(ll11)
print('✓ γ 就是“每帧该按多少权重向各符号收敛”——CTC 梯度的分母来源')

扩展序列 = [0 1 0 2 0]
logP(标签|语音) = -5.8037
每帧 γ 总和 = [1. 1. 1. 1. 1.] （每帧 = 1：概率守恒）
结论：首尾帧贡献小、中间帧贡献大；无逐帧标签，但每帧都有软监督梯度信号
✓ γ 就是“每帧该按多少权重向各符号收敛”——CTC 梯度的分母来源


In [43]:
# Attention 权重热图：3 个输出 token 对齐 5 帧编码（软对齐可视化）
W12 = np.array([[0.9, 0.1, 0.0, 0.0, 0.0],
               [0.1, 0.5, 0.4, 0.0, 0.0],
               [0.0, 0.1, 0.2, 0.5, 0.2]])
plt.figure(figsize=(6.5, 3.0))
plt.imshow(W12, aspect='auto', cmap='Blues', vmin=0, vmax=1)
for t in range(3):
    for i in range(5):
        plt.text(i, t, '%.1f' % W12[t, i], ha='center', va='center', fontsize=8)
plt.colorbar(label='α (注意力权重)')
plt.yticks(range(3), ['出:我', '出:爱', '出:你'])
plt.xticks(range(5), ['编码1', '编码2', '编码3', '编码4', '编码5'])
plt.title('软对齐：每行（每个输出字）对所有帧加权，行和=1', fontsize=10)
plt.tight_layout(); plt.show()
assert np.allclose(W12.sum(axis=1), 1)
print('✓ 行和=1 是 softmax 性质；注意力逐步右移 = 单调对齐（读字顺序不乱）')

✓ 行和=1 是 softmax 性质；注意力逐步右移 = 单调对齐（读字顺序不乱）


C:\Users\24260\AppData\Local\Temp\ipykernel_50688\1276404979.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⚠️ 本章高频踩坑

1. **概率域 vs log 域**：手算 P 与 DP 的 logP 别直接相减——先 exp 再比（本书 r9 的真实事故）；
2. **束宽 ≠ 全枚举**：束宽=词表时才近似精确（路径数是 V^T，不是 V）；
3. **同标签不能连读**：必须 blank 隔开——问“为什么”要答“折叠去重”；
4. **帧独立性假设**：CTC 假设帧间独立 → 无语言记忆 → 常需外部 LM 或 AED 化；
5. **注意力崩溃**：AED 长句可能“注意力漂移/漏字”→ 单调/位置约束；
6. **训练/推理不一致**：训练 teacher forcing、推理自回归 → 曝光偏差 → 计划采样/蒸馏缓解；
7. **流式改造**：整句 Encoder 不能流式 → 分块/单调注意力/RNN-T；
8. **解码长度**：AED 需要显式 EOS；beam 搜索要有长度惩罚防短句偏好。

## 🧭 四代端到端对比：CTC / AED / RNN-T / Whisper

| 模型 | 对齐方式 | 语言建模 | 流式 | 代表 |
|---|---|---|---|---|
| CTC | 显式格子+blank | ✗（需外部 LM） | ✓ | 百度 DeepSpeech |
| AED | 软注意力 | ✓（Decoder 自带） | ✗ | LAS、Joint CTC+Att |
| RNN-T | 联合网络边走边定 | ✓ | ✓ | 大厂流式主力 |
| Whisper | AED+大规模多语 | ✓ | ✗ | OpenAI 通用模型 |

**选型逻辑**：
- 要**实时**（客服/字幕）→ CTC 或 RNN-T；
- 要**最高精度**（离线转写）→ AED/Whisper 风格；
- 要**低资源**（嵌入式）→ CTC 最省（解码 = argmax+collapse）；
- 要**多语/通用** → 大规模预训练 AED。

**融合技巧**：Joint CTC-Attention 用 CTC 分数做解码引导，兼顾独立性与语言建模。

## 🏁 本章学完清单

- [ ] 能用“老师只给答案”讲清 CTC 原理；
- [ ] 能解释 blank 为什么必须存在（重复帧/静音吸收）；
- [ ] 能手写 P(ℓ|X)=ΣP(π|X) 并解释每个符号；
- [ ] 能说出前向递推三项来路与 s-2 的合法性条件；
- [ ] 能解释为什么 CTC 每帧都有梯度（软监督）；
- [ ] 能写出 Attention 三公式并画出软对齐热图；
- [ ] 能对比 CTC/AED/RNN-T/Whisper 的取舍；
- [ ] 能徒手算 3 帧 3 符号的 CTC 前向与枚举（r9 示例）。

> 下一站：TTS——把文字变回声音（下一本 notebook）。